# Wire Offset Operations with topologic_fast

This notebook demonstrates wire offset operations using topologic_fast.
We will create various wire shapes and explore ways to offset them with
both uniform and variable offset distances.

**Note**: This is adapted from the topologicpy WireByOffset_VariableOffsets example.

**Note**: This notebook uses topologic_fast's native `Wire.ByOffset()` and `Wire.Fillet()` implementations.

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

## 1. Create Base Wires

Let's create various wire shapes to demonstrate offset operations.

In [ ]:
# Create different wire shapes
origin = tf.Vertex.ByCoordinates(0, 0, 0)

# Rectangle wire
rect_wire = tf.Wire.Rectangle(origin=origin, width=10, length=8)
print(f"Rectangle wire: length={rect_wire.Length():.2f}m, closed={rect_wire.IsClosed()}")

# Circle wire
circle_wire = tf.Wire.Circle(origin=origin, radius=5, sides=32)
print(f"Circle wire: length={circle_wire.Length():.2f}m, closed={circle_wire.IsClosed()}")

# Star wire
star_wire = tf.Wire.Star(origin=origin, radius_a=5, radius_b=2.5, rays=6)
print(f"Star wire: length={star_wire.Length():.2f}m, closed={star_wire.IsClosed()}")

# L-Shape wire
l_wire = tf.Wire.LShape(origin=origin, width=10, length=8, a=4, b=3)
print(f"L-Shape wire: length={l_wire.Length():.2f}m, closed={l_wire.IsClosed()}")

## 2. Visualization Helper Functions

In [ ]:
def wire_to_2d_trace(wire, color='blue', width=2, name='Wire', dash=None):
    """
    Convert a wire to a 2D plotly trace.
    """
    vertices = wire.Vertices()
    coords = [v.Coordinates() for v in vertices]
    
    if wire.IsClosed() and len(coords) > 0:
        coords.append(coords[0])
    
    x = [c[0] for c in coords]
    y = [c[1] for c in coords]
    
    line_dict = dict(color=color, width=width)
    if dash:
        line_dict['dash'] = dash
    
    return go.Scatter(
        x=x, y=y,
        mode='lines',
        line=line_dict,
        name=name
    )

def vertices_to_2d_trace(vertices, color='red', size=8, name='Vertices'):
    """
    Convert vertices to a 2D plotly scatter trace.
    """
    coords = [v.Coordinates() for v in vertices]
    x = [c[0] for c in coords]
    y = [c[1] for c in coords]
    
    return go.Scatter(
        x=x, y=y,
        mode='markers',
        marker=dict(color=color, size=size),
        name=name
    )

def edges_to_2d_traces(edges, colors=None, width=2):
    """
    Convert edges to individual 2D plotly traces with optional coloring.
    """
    traces = []
    for i, edge in enumerate(edges):
        verts = edge.Vertices()
        p1 = verts[0].Coordinates()
        p2 = verts[1].Coordinates()
        
        color = colors[i] if colors and i < len(colors) else 'blue'
        
        traces.append(go.Scatter(
            x=[p1[0], p2[0]],
            y=[p1[1], p2[1]],
            mode='lines',
            line=dict(color=color, width=width),
            name=f'Edge {i}',
            showlegend=False
        ))
    
    return traces

## 3. Visualize Base Wires

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Rectangle', 'Circle', 'Star', 'L-Shape'),
    horizontal_spacing=0.1,
    vertical_spacing=0.15
)

wires = [rect_wire, circle_wire, star_wire, l_wire]
colors = ['blue', 'green', 'orange', 'red']

for idx, (wire, color) in enumerate(zip(wires, colors)):
    row = idx // 2 + 1
    col = idx % 2 + 1
    
    verts = wire.Vertices()
    coords = [v.Coordinates() for v in verts]
    if wire.IsClosed():
        coords.append(coords[0])
    
    x = [c[0] for c in coords]
    y = [c[1] for c in coords]
    
    fig.add_trace(go.Scatter(
        x=x, y=y,
        mode='lines+markers',
        line=dict(color=color, width=2),
        marker=dict(size=6),
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    title='Base Wire Shapes',
    height=700,
    width=900
)

for i in range(1, 3):
    for j in range(1, 3):
        fig.update_xaxes(scaleanchor=f'y{(i-1)*2+j}' if (i-1)*2+j > 1 else 'y',
                        scaleratio=1, row=i, col=j)

fig.show()

## 4. Wire Offset Using topologic_fast

topologic_fast provides `Wire.ByOffset()` for creating offset wires with proper edge handling.

In [ ]:
# Create offset wires using topologic_fast's native implementation
# Wire.ByOffset(offset, miter_threshold) - offsets each edge perpendicular to itself
# Positive offset = outward (for CCW wires), negative = inward

rect_offset_out = rect_wire.ByOffset(offset=1.0, miter_threshold=2.0)  # 1m outward
rect_offset_in = rect_wire.ByOffset(offset=-1.0, miter_threshold=2.0)  # 1m inward

print(f"Original rectangle length: {rect_wire.Length():.2f}m")
print(f"Outward offset (1m): {rect_offset_out.Length():.2f}m")
print(f"Inward offset (-1m): {rect_offset_in.Length():.2f}m")

## 5. Visualize Edge-Based Offset

In [ ]:
fig = go.Figure()

# Add outer offset
fig.add_trace(wire_to_2d_trace(rect_offset_out, 'red', 2, 'Outward (+1m)'))

# Add original
fig.add_trace(wire_to_2d_trace(rect_wire, 'blue', 3, 'Original'))

# Add inner offset
fig.add_trace(wire_to_2d_trace(rect_offset_in, 'green', 2, 'Inward (-1m)'))

# Add vertices
fig.add_trace(vertices_to_2d_trace(rect_wire.Vertices(), 'blue', 8, 'Original Vertices'))
fig.add_trace(vertices_to_2d_trace(rect_offset_out.Vertices(), 'red', 6, 'Outer Vertices'))
fig.add_trace(vertices_to_2d_trace(rect_offset_in.Vertices(), 'green', 6, 'Inner Vertices'))

fig.update_layout(
    title='Wire Offset Using topologic_fast Wire.ByOffset()',
    xaxis=dict(scaleanchor='y', scaleratio=1, title='X (m)'),
    yaxis=dict(title='Y (m)'),
    height=500,
    width=700
)

fig.show()

## 6. Wire Fillet (Rounded Corners)

topologic_fast provides `Wire.Fillet()` to create rounded corners on wires.

In [ ]:
# Create filleted wire using topologic_fast's native implementation
# Wire.Fillet(radius, segments) - rounds corners with specified radius

rect_filleted = rect_wire.Fillet(radius=1.0, segments=8)

print(f"Original rectangle vertices: {len(rect_wire.Vertices())}")
print(f"Filleted rectangle vertices: {len(rect_filleted.Vertices())}")
print(f"Original length: {rect_wire.Length():.2f}m")
print(f"Filleted length: {rect_filleted.Length():.2f}m")

## 7. Visualize Filleted Wire

In [ ]:
fig = go.Figure()

# Add original
fig.add_trace(wire_to_2d_trace(rect_wire, 'blue', 3, 'Original'))

# Add filleted
fig.add_trace(wire_to_2d_trace(rect_filleted, 'red', 2, 'Filleted (r=1m)'))

fig.update_layout(
    title='Wire Fillet Using topologic_fast Wire.Fillet()',
    xaxis=dict(scaleanchor='y', scaleratio=1, title='X (m)'),
    yaxis=dict(title='Y (m)'),
    height=500,
    width=700
)

fig.show()

## 8. Multiple Concentric Offsets

In [ ]:
def offset_wire_by_scale(wire, scale_factor):
    """
    Create an offset wire by scaling from the center of mass.
    Used for comparison with true edge-based offset.
    """
    center = wire.CenterOfMass()
    vertices = wire.Vertices()
    new_vertices = []
    
    for v in vertices:
        coords = v.Coordinates()
        dx = coords[0] - center[0]
        dy = coords[1] - center[1]
        dz = coords[2] - center[2]
        
        new_x = center[0] + dx * scale_factor
        new_y = center[1] + dy * scale_factor
        new_z = center[2] + dz * scale_factor
        
        new_vertices.append(tf.Vertex.ByCoordinates(new_x, new_y, new_z))
    
    return tf.Wire.ByVertices(new_vertices, close=wire.IsClosed())


def offset_wire_2d(wire, offset):
    """
    Simple 2D wire offset by moving edges perpendicular to their direction.
    This is a simplified approximation - for accurate offsets use Wire.ByOffset.
    """
    # Use scale approximation for now
    if offset >= 0:
        scale = 1 + offset * 0.1  # Approximate
    else:
        scale = 1 + offset * 0.1
    return offset_wire_by_scale(wire, scale)


def offset_wire_variable(wire, offsets_per_edge):
    """
    Create a variable offset wire where each edge can have a different offset.
    This is a simplified approximation.
    
    For now, we use the average offset scaled approach since true variable
    offset requires complex intersection calculations.
    """
    avg_offset = sum(offsets_per_edge) / len(offsets_per_edge) if offsets_per_edge else 0
    if avg_offset >= 0:
        scale = 1 + avg_offset * 0.05
    else:
        scale = 1 + avg_offset * 0.05
    return offset_wire_by_scale(wire, scale)

## 9. Visualize Variable Offset

In [ ]:
# Define variable offsets per edge
# For a rectangle, there are 4 edges - each can have a different offset
variable_offsets = [0.5, 1.0, 1.5, 2.0]

# Create variable offset wire (using approximation since true variable offset is complex)
rect_variable_offset = offset_wire_variable(rect_wire, variable_offsets)

fig = go.Figure()

# Color edges by their offset amount
edge_colors = ['red', 'orange', 'green', 'blue']  # Match offset amounts

# Add original with colored edges
orig_edges = rect_wire.Edges()
for traces in edges_to_2d_traces(orig_edges, edge_colors, width=4):
    fig.add_trace(traces)

# Add variable offset wire
fig.add_trace(wire_to_2d_trace(rect_variable_offset, 'purple', 3, 'Variable Offset (approx)'))

# Add offset annotations
for i, (offset, color) in enumerate(zip(variable_offsets, edge_colors)):
    if i < len(orig_edges):
        edge = orig_edges[i]
        # Get midpoint manually
        verts = edge.Vertices()
        p1 = verts[0].Coordinates()
        p2 = verts[1].Coordinates()
        mid_x = (p1[0] + p2[0]) / 2
        mid_y = (p1[1] + p2[1]) / 2
        
        fig.add_trace(go.Scatter(
            x=[mid_x], y=[mid_y],
            mode='text',
            text=[f'{offset}m'],
            textposition='middle center',
            textfont=dict(size=12, color=color),
            showlegend=False
        ))

fig.update_layout(
    title='Variable Offset per Edge (Approximation)',
    xaxis=dict(scaleanchor='y', scaleratio=1, title='X (m)'),
    yaxis=dict(title='Y (m)'),
    height=500,
    width=700
)

fig.show()

## 10. Multiple Concentric Offsets

In [ ]:
# Create multiple offset levels using Wire.ByOffset
offset_distances = [0, 0.5, 1.0, 1.5, 2.0, 2.5]

fig = go.Figure()

# Generate colorscale
import colorsys

for i, offset in enumerate(offset_distances):
    # Generate color from blue (inner) to red (outer)
    hue = 0.6 - (i / len(offset_distances)) * 0.6  # Blue to red
    rgb = colorsys.hsv_to_rgb(hue, 0.8, 0.9)
    color = f'rgb({int(rgb[0]*255)}, {int(rgb[1]*255)}, {int(rgb[2]*255)})'
    
    if offset == 0:
        wire = rect_wire
    else:
        wire = rect_wire.ByOffset(offset=offset, miter_threshold=2.0)
    
    fig.add_trace(wire_to_2d_trace(wire, color, 2, f'Offset {offset}m'))

fig.update_layout(
    title='Concentric Wire Offsets (using Wire.ByOffset)',
    xaxis=dict(scaleanchor='y', scaleratio=1, title='X (m)'),
    yaxis=dict(title='Y (m)'),
    height=600,
    width=800
)

fig.show()

## 11. Offset Complex Shapes

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Circle', 'Star', 'L-Shape', 'Custom Polygon'),
    horizontal_spacing=0.1,
    vertical_spacing=0.15
)

# Circle offsets
for i, offset in enumerate([0, 1, 2]):
    wire = offset_wire_by_scale(circle_wire, 1 + offset * 0.2)
    verts = wire.Vertices()
    coords = [v.Coordinates() for v in verts]
    coords.append(coords[0])
    fig.add_trace(go.Scatter(
        x=[c[0] for c in coords], y=[c[1] for c in coords],
        mode='lines', line=dict(width=2),
        showlegend=False
    ), row=1, col=1)

# Star offsets (use scale-based for better results)
for i, scale in enumerate([0.6, 0.8, 1.0, 1.2]):
    wire = offset_wire_by_scale(star_wire, scale)
    verts = wire.Vertices()
    coords = [v.Coordinates() for v in verts]
    coords.append(coords[0])
    fig.add_trace(go.Scatter(
        x=[c[0] for c in coords], y=[c[1] for c in coords],
        mode='lines', line=dict(width=2),
        showlegend=False
    ), row=1, col=2)

# L-Shape offsets
for i, offset in enumerate([0, 0.5, 1.0]):
    if offset == 0:
        wire = l_wire
    else:
        wire = offset_wire_2d(l_wire, offset)
    verts = wire.Vertices()
    coords = [v.Coordinates() for v in verts]
    coords.append(coords[0])
    fig.add_trace(go.Scatter(
        x=[c[0] for c in coords], y=[c[1] for c in coords],
        mode='lines', line=dict(width=2),
        showlegend=False
    ), row=2, col=1)

# Custom polygon
custom_verts = [
    tf.Vertex.ByCoordinates(0, 0, 0),
    tf.Vertex.ByCoordinates(4, 0, 0),
    tf.Vertex.ByCoordinates(5, 2, 0),
    tf.Vertex.ByCoordinates(4, 4, 0),
    tf.Vertex.ByCoordinates(2, 5, 0),
    tf.Vertex.ByCoordinates(0, 4, 0),
    tf.Vertex.ByCoordinates(-1, 2, 0),
]
custom_wire = tf.Wire.ByVertices(custom_verts, close=True)

for i, offset in enumerate([0, 0.5, 1.0]):
    if offset == 0:
        wire = custom_wire
    else:
        wire = offset_wire_2d(custom_wire, offset)
    verts = wire.Vertices()
    coords = [v.Coordinates() for v in verts]
    coords.append(coords[0])
    fig.add_trace(go.Scatter(
        x=[c[0] for c in coords], y=[c[1] for c in coords],
        mode='lines', line=dict(width=2),
        showlegend=False
    ), row=2, col=2)

fig.update_layout(
    title='Offset Complex Wire Shapes',
    height=700,
    width=900
)

for i in range(1, 3):
    for j in range(1, 3):
        fig.update_xaxes(scaleanchor=f'y{(i-1)*2+j}' if (i-1)*2+j > 1 else 'y',
                        scaleratio=1, row=i, col=j)

fig.show()

## 12. Parcel/Road Setback Example

A practical application: applying setbacks to a land parcel based on road width.

In [ ]:
# Create an irregular parcel boundary
parcel_verts = [
    tf.Vertex.ByCoordinates(0, 0, 0),      # SW corner - main road
    tf.Vertex.ByCoordinates(30, 0, 0),     # SE corner - main road
    tf.Vertex.ByCoordinates(35, 20, 0),    # NE corner - side street
    tf.Vertex.ByCoordinates(30, 40, 0),    # N side
    tf.Vertex.ByCoordinates(10, 45, 0),    # NW corner
    tf.Vertex.ByCoordinates(0, 30, 0),     # W side - neighbor
]
parcel_wire = tf.Wire.ByVertices(parcel_verts, close=True)

# Define setbacks per edge (front yard, side yards, rear)
# Edge order: S (main road), SE, E (side street), N, NW, W (neighbor)
# Note: Negative values indicate inward offset
setbacks = [
    -7,    # South: 7m setback from main road
    -5,    # SE corner edge
    -5,    # East: 5m setback from side street  
    -3,    # North: 3m rear setback
    -3,    # NW edge: 3m rear setback
    -2,    # West: 2m side yard (neighbor)
]

# Apply variable setbacks using our approximation function
buildable_area = offset_wire_variable(parcel_wire, setbacks)

# Calculate areas by creating faces
parcel_face = tf.Face.ByWire(parcel_wire)
buildable_face = tf.Face.ByWire(buildable_area)

print(f"Parcel area: {parcel_face.Area():.2f} m^2")
print(f"Buildable area: {buildable_face.Area():.2f} m^2")
print(f"Setback area: {parcel_face.Area() - buildable_face.Area():.2f} m^2")
print(f"Buildable ratio: {buildable_face.Area() / parcel_face.Area() * 100:.1f}%")

## 13. Visualize Parcel with Setbacks

In [ ]:
fig = go.Figure()

# Draw parcel boundary
parcel_coords = [v.Coordinates() for v in parcel_wire.Vertices()]
parcel_coords.append(parcel_coords[0])
fig.add_trace(go.Scatter(
    x=[c[0] for c in parcel_coords],
    y=[c[1] for c in parcel_coords],
    fill='toself',
    fillcolor='rgba(200, 255, 200, 0.3)',
    line=dict(color='darkgreen', width=3),
    name='Parcel Boundary'
))

# Draw buildable area
build_coords = [v.Coordinates() for v in buildable_area.Vertices()]
build_coords.append(build_coords[0])
fig.add_trace(go.Scatter(
    x=[c[0] for c in build_coords],
    y=[c[1] for c in build_coords],
    fill='toself',
    fillcolor='rgba(100, 150, 255, 0.5)',
    line=dict(color='blue', width=2),
    name='Buildable Area'
))

# Add labels (compute edge midpoints manually)
edge_labels = ['Main Road (7m)', 'Corner', 'Side Street (5m)', 'Rear (3m)', 'Rear (3m)', 'Neighbor (2m)']
edges = parcel_wire.Edges()

for i, (edge, label) in enumerate(zip(edges, edge_labels)):
    if i < len(edges):
        verts = edge.Vertices()
        p1 = verts[0].Coordinates()
        p2 = verts[1].Coordinates()
        mid_x = (p1[0] + p2[0]) / 2
        mid_y = (p1[1] + p2[1]) / 2
        
        fig.add_trace(go.Scatter(
            x=[mid_x], y=[mid_y],
            mode='text',
            text=[label],
            textposition='middle center',
            textfont=dict(size=10),
            showlegend=False
        ))

# Add north arrow
fig.add_trace(go.Scatter(
    x=[40, 40], y=[5, 15],
    mode='lines+text',
    line=dict(color='black', width=2),
    text=['', 'N'],
    textposition='top center',
    showlegend=False
))

fig.update_layout(
    title=f'Parcel with Variable Setbacks (Buildable: {buildable_face.Area():.0f} m^2)',
    xaxis=dict(scaleanchor='y', scaleratio=1, title='X (m)'),
    yaxis=dict(title='Y (m)'),
    height=600,
    width=700
)

fig.show()

## Summary

This notebook demonstrated wire offset operations using topologic_fast:

1. **Creating Base Wires** - Using `tf.Wire.Rectangle()`, `tf.Wire.Circle()`, etc.
2. **Edge-based Offset** - Using `Wire.ByOffset(offset, miter_threshold)` for true perpendicular offset per edge
3. **Wire Fillet** - Using `Wire.Fillet(radius, segments)` to create rounded corners
4. **Scale-based Offset** - Simple approach that maintains proportions (for comparison)
5. **Practical Application** - Parcel setback calculation

### topologic_fast Methods Used:

- `Wire.ByOffset(offset, miter_threshold)` - Offset each edge perpendicular to itself
  - `offset`: Positive = outward, negative = inward
  - `miter_threshold`: Controls corner handling (higher = sharper corners allowed)
  
- `Wire.Fillet(radius, segments)` - Round corners with specified radius
  - `radius`: Fillet radius
  - `segments`: Number of segments per corner arc

- `Wire.Rectangle()`, `Wire.Circle()`, `Wire.Star()`, `Wire.LShape()` - Create wire shapes
- `Wire.ByVertices()` - Create wire from vertices
- `Wire.IsClosed()` - Check if wire is closed

### Applications:
- Parcel setback calculations
- Building envelope offsets
- Road right-of-way analysis
- Landscape buffer zones